# Pruebas API Registro de Horas

Este notebook prueba los endpoints `/registro-horas` y `/registro-horas/export`.

In [1]:
import json
from datetime import date
from pathlib import Path

import requests

In [2]:
BASE_URL = "http://localhost:8000"

In [3]:
def request_json(method, path, payload=None, params=None):
    url = f"{BASE_URL}{path}"
    response = requests.request(method, url, json=payload, params=params)
    try:
        data = response.json()
    except ValueError:
        data = response.text
    return response.status_code, data


def pretty(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))

In [5]:
empleado_payload = {
    "dni": "99999999",
    "nombre_completo": "Joaquin Amaya",
    "observaciones": "Prueba registro horas",
}

status, empleados = request_json("GET", "/empleados")
if status != 200:
    raise RuntimeError(f"Error al listar empleados: {status} - {empleados}")

empleado = next((e for e in empleados if e["dni"] == empleado_payload["dni"]), None)
if not empleado:
    status, empleado = request_json("POST", "/empleados", payload=empleado_payload)
    if status not in (200, 201):
        raise RuntimeError(f"Error al crear empleado: {status} - {empleado}")

empleado_id = empleado["id"]
print("Empleado:", empleado_id)

Empleado: 4


In [6]:
cliente_payload = {
    "nombre": "Coca Cola",
    "direccion": "Av. Corrientes 1746",
}

status, clientes = request_json("GET", "/clientes", params={"search": cliente_payload["nombre"]})
if status != 200:
    raise RuntimeError(f"Error al buscar clientes: {status} - {clientes}")

cliente = next((c for c in clientes if c["nombre"].lower() == cliente_payload["nombre"].lower()), None)
if not cliente:
    status, cliente = request_json("POST", "/clientes", payload=cliente_payload)
    if status not in (200, 201):
        raise RuntimeError(f"Error al crear cliente: {status} - {cliente}")

cliente_id = cliente["id"]
print("Cliente:", cliente_id)

Cliente: 2


In [7]:
fecha_hoy = date.today().isoformat()

jornada_payload = {
    "fecha": fecha_hoy,
    "hora_inicio": "08:00:00",
    "hora_fin": "12:00:00",
    "cliente": "Coca Cola",
    "direccion": "Av. Corrientes 1746",
    "descripcion": "Jornada de prueba",
    "empleado_id": empleado_id,
}

status, jornada = request_json("POST", "/jornadas", payload=jornada_payload)
if status not in (200, 201, 409):
    raise RuntimeError(f"Error al crear jornada: {status} - {jornada}")

print("Jornada creada o ya existente.")

Jornada creada o ya existente.


In [8]:
asistencia_entrada = {
    "tipo": "entrada",
    "dni": empleado_payload["dni"],
    "cliente_id": cliente_id,
    "direccion": "Av. Corrientes 1746",
}

asistencia_salida = {
    "tipo": "salida",
    "dni": empleado_payload["dni"],
    "cliente_id": cliente_id,
    "direccion": "Av. Corrientes 1746",
}

status, respuesta = request_json("POST", "/asistencias", payload=asistencia_entrada)
if status not in (200, 201, 409):
    raise RuntimeError(f"Error al registrar entrada: {status} - {respuesta}")

status, respuesta = request_json("POST", "/asistencias", payload=asistencia_salida)
if status not in (200, 201, 409):
    raise RuntimeError(f"Error al registrar salida: {status} - {respuesta}")

print("Asistencias registradas o ya existentes.")

Asistencias registradas o ya existentes.


In [9]:
status, registros = request_json(
    "GET",
    "/registro-horas",
    params={
        "fecha": fecha_hoy,
        "empleado": "Joaquin",
        "cliente": "Coca Cola",
    },
)

print("Status:", status)
pretty(registros)

Status: 200
[
  {
    "fecha": "2026-06-10",
    "empleado": "Joaquin Amaya",
    "cliente": "Coca Cola",
    "direccion": "Av. Corrientes 1746",
    "hora_entrada": "16:30",
    "hora_salida": "16:30",
    "horas_estimadas": 4.0,
    "horas_realizadas": 0.0,
    "horas_extras": 0.0,
    "horas_a_descontar": 4.0,
    "estado": null,
    "message": null
  }
]


In [10]:
status, registros = request_json(
    "GET",
    "/registro-horas",
    params={
        "search": "Corrientes",
    },
)

print("Status:", status)
pretty(registros)

Status: 200
[
  {
    "fecha": "2026-06-10",
    "empleado": "Joaquin Amaya",
    "cliente": "Coca Cola",
    "direccion": "Av. Corrientes 1746",
    "hora_entrada": "16:30",
    "hora_salida": "16:30",
    "horas_estimadas": 4.0,
    "horas_realizadas": 0.0,
    "horas_extras": 0.0,
    "horas_a_descontar": 4.0,
    "estado": null,
    "message": null
  },
  {
    "fecha": "2026-06-03",
    "empleado": "Rogelio Johann",
    "cliente": "Cliente Ejemplo S.A.",
    "direccion": "Av. Corrientes 1234, CABA",
    "hora_entrada": null,
    "hora_salida": null,
    "horas_estimadas": 9.0,
    "horas_realizadas": 0.0,
    "horas_extras": 0.0,
    "horas_a_descontar": 0.0,
    "estado": "incompleta",
    "message": "Sin asistencia registrada"
  }
]


In [11]:
params = {"fecha": fecha_hoy, "empleado": "Joaquin"}
response = requests.get(f"{BASE_URL}/registro-horas/export", params=params)
response.raise_for_status()

archivo = Path("registro_horas.xlsx")
archivo.write_bytes(response.content)
print("Archivo generado:", archivo.resolve())

Archivo generado: C:\Users\Lenovo\Desktop\REPO-CRUD\CRUD-FastAPI\backend\registro_horas.xlsx
